# Justinnn

In [7]:
change_cols = change_feature_cols=[  # specify the change feature columns
    "tech_investment_ratio_change",
    "nib_deposit_ratio_change",
    "service_charge_intensity_change",
    "efficiency_ratio_change",
    "nonint_income_pct_change",
    "loans_to_assets_change",
    "equity_to_assets_change",
    "deposits_to_assets_change",
    "roa_change",
    "roe_change",
    "nontrans_deposits_pct_change",
    "digital_revenue_ratio_change",
    "non_branch_revenue_pct_change",
    "loan_yield_change",
    "securities_to_assets_change",
    "expense_per_salary_dollar_change",
    "occupancy_intensity_change",
    "chargeoff_rate_change",
    "provision_intensity_change",
]
from bank_innovation.clustering import cluster_by_tier, comprehensive_tuning, cluster_with_best_params, assign_cluster_names
from bank_innovation.analysis import analyze_innovation_change_clusters
from bank_innovation.migration import build_bank_cluster_history, detect_migrations
from bank_innovation.visualization import create_clustering_visualization
import pandas as pd

In [ ]:
from bank_innovation.visualization import create_clustering_visualization, print_clustering_summary
from bank_innovation.analysis import analyze_innovation_change_clusters
df_changes_optimized = pd.read_csv('data/bank_innovation_clustered_optimized.csv')
print_clustering_summary(df_changes_optimized)
create_clustering_visualization(df_changes_optimized, save_path='viz/3yr_window_clustering.png')

for tier in ['Large', 'Medium', 'Small']:
    analyze_innovation_change_clusters(df_changes_optimized, tier, change_feature_cols)

In [ ]:
for tier in ['Large', 'Medium', 'Small']:
    print(f"\n=== {tier} Tier Cluster Analysis ===")
    print(analyze_innovation_change_clusters(df_changes_optimized, tier, change_feature_cols))

In [ ]:
for tier in ['Large', 'Medium', 'Small']:
    tier_data = df_changes_optimized[df_changes_optimized['bank_tier'] == tier]
    clusters = sorted(c for c in tier_data['innovation_cluster'].unique() if c != -1)
    
    # Get top 5 features for this tier
    top_feats = analyze_innovation_change_clusters(
        df_changes_optimized, tier, change_feature_cols
    ).head(5)['feature'].tolist()
    
    print(f"\n{'='*70}")
    print(f"{tier} — Cluster means for top 5 features")
    print(f"{'='*70}")
    
    for feat in top_feats:
        print(f"\n  {feat.replace('_change', '')}:")
        for cl in clusters:
            cl_data = tier_data[tier_data['innovation_cluster'] == cl]
            print(f"    Cluster {cl} (n={len(cl_data):>5}): "
                  f"mean={cl_data[feat].mean():>10.3f}  std={cl_data[feat].std():>10.3f}")

In [13]:
df_changes_optimized.to_csv('data/bank_innovation_clustered_optimized.csv', index=False)

In [14]:
from bank_innovation.migration import (
    build_bank_cluster_history,
    detect_migrations,
    compute_migration_matrix,
    compute_migration_rates_by_year,
    summarize_bank_trajectories,
)

history = build_bank_cluster_history(df_changes_optimized)
migrations = detect_migrations(df_changes_optimized)

print(f"\nMigration events: {len(migrations):,}")
print(f"Banks that migrated: {migrations['rssd9017'].nunique():,}")

# Migration rates over time
for tier in ['Large', 'Medium', 'Small']:
    rates = compute_migration_rates_by_year(df_changes_optimized, migrations, tier=tier)
    print(f"\n{tier} migration rates:")
    print(rates.to_string(index=False))

# Transition matrices
for tier in ['Large', 'Medium', 'Small']:
    print(f"\n{tier} transition matrix:")
    matrix = compute_migration_matrix(migrations, tier=tier)
    print(matrix.round(3).to_string())

# Bank trajectories
for tier in ['Large', 'Medium', 'Small']:
    traj = summarize_bank_trajectories(history, tier=tier)
    movers = traj[traj['n_migrations'] > 0].sort_values('n_migrations', ascending=False)
    print(f"\n{tier} — top 10 most active migrators:")
    print(movers.head(10)[['rssd9017', 'trajectory', 'n_migrations']].to_string(index=False))


Migration events: 5,253
Banks that migrated: 2,623

Large migration rates:
 year  total_banks  migrating_banks  migration_rate
 2013           75                0        0.000000
 2014           82               23        0.280488
 2015           96               18        0.187500
 2016           95               18        0.189474
 2017          102               16        0.156863
 2018          107               18        0.168224
 2019          116               24        0.206897
 2020          112               29        0.258929
 2021          122               50        0.409836

Medium migration rates:
 year  total_banks  migrating_banks  migration_rate
 2013          378                0        0.000000
 2014          398               47        0.118090
 2015          460               46        0.100000
 2016          478               59        0.123431
 2017          493               77        0.156187
 2018          502              105        0.209163
 2019          

In [5]:
for tier in ['Large', 'Medium', 'Small']:
    tier_data = df_changes_optimized[df_changes_optimized['bank_tier'] == tier]
    clusters = sorted(c for c in tier_data['innovation_cluster'].unique() if c != -1)
    
    print(f"\n{'='*70}")
    print(f"{tier} — Cluster sizes by window end year (year_to)")
    print(f"{'='*70}")
    
    ct = pd.crosstab(
        tier_data['year_to'], 
        tier_data['innovation_cluster']
    )
    # Drop noise column if present
    if -1 in ct.columns:
        ct = ct.drop(columns=-1)
    
    print(ct.to_string())
    
    # Also show top 3 features per cluster per window period
    # Split into early (2013-2016) vs late (2017-2021)
    print(f"\n  Early (year_to <= 2016) vs Late (year_to >= 2017):")
    
    top_feats = analyze_innovation_change_clusters(
        df_changes_optimized, tier, change_feature_cols
    ).head(3)['feature'].tolist()
    
    for feat in top_feats:
        print(f"\n  {feat.replace('_change', '')}:")
        for cl in clusters:
            cl_data = tier_data[tier_data['innovation_cluster'] == cl]
            early = cl_data[cl_data['year_to'] <= 2016][feat]
            late = cl_data[cl_data['year_to'] >= 2017][feat]
            print(f"    Cluster {cl}: early={early.mean():>8.2f} (n={len(early):>3})  "
                  f"late={late.mean():>8.2f} (n={len(late):>3})")


Large — Cluster sizes by window end year (year_to)
innovation_cluster   0  1    2   3
year_to                           
2013                 2  3   59   5
2014                 0  3   70   2
2015                 0  1   83   3
2016                 0  1   93   0
2017                 0  2   95   0
2018                 0  5  101   0
2019                 0  6  106   0
2020                 6  3   87   4
2021                27  0   69  23

  Early (year_to <= 2016) vs Late (year_to >= 2017):

  nontrans_deposits_pct:
    Cluster 0: early=   36.56 (n=  2)  late=   39.10 (n= 33)
    Cluster 1: early=   -1.29 (n=  8)  late=    1.09 (n= 16)
    Cluster 2: early=    1.57 (n=305)  late=    0.84 (n=458)
    Cluster 3: early=    4.45 (n= 10)  late=    9.76 (n= 27)

  digital_revenue_ratio:
    Cluster 0: early=    0.48 (n=  2)  late=    0.17 (n= 33)
    Cluster 1: early= -149.81 (n=  8)  late= -116.67 (n= 16)
    Cluster 2: early=   -3.07 (n=305)  late=   -1.05 (n=458)
    Cluster 3: early=   -1.88 

In [ ]:
# Reorder clusters and apply names
reorder_map = {
    'Large': {3: 0, 0: 1, 2: 2, 1: 3},
    'Medium': {0: 0, 2: 1, 4: 2, 1: 3, 3: 4},
    'Small': {3: 0, 0: 1, 1: 2, 2: 3},
}

for tier, mapping in reorder_map.items():
    mask = (df_changes_optimized['bank_tier'] == tier) & (df_changes_optimized['innovation_cluster'] != -1)
    df_changes_optimized.loc[mask, 'innovation_cluster'] = (
        df_changes_optimized.loc[mask, 'innovation_cluster'].map(mapping)
    )

cluster_names = {
    'Large': {
        -1: 'Noise',
        0: 'Branch Consolidators',
        1: 'Deposit Gatherers',
        2: 'Steady Operators',
        3: 'Digital Retreaters',
    },
    'Medium': {
        -1: 'Noise',
        0: 'Branch Expanders',
        1: 'Deposit Restructurers',
        2: 'Deposit Transformers',
        3: 'Steady Operators',
        4: 'Relationship Builders',
    },
    'Small': {
        -1: 'Noise',
        0: 'Rapid Capitalizers',
        1: 'Funding Diversifiers',
        2: 'Steady Operators',
        3: 'Stressed Performers',
        
    },
}

df_changes_optimized, _ = assign_cluster_names(df_changes_optimized, cluster_names)

# Verify
for tier in ['Large', 'Medium', 'Small']:
    print(f"\n{tier}:")
    tier_data = df_changes_optimized[df_changes_optimized['bank_tier'] == tier]
    print(tier_data.groupby(['innovation_cluster', 'cluster_name']).size().to_string())


Large:
innovation_cluster  cluster_name        
-1                  Noise                    48
 0                  Branch Consolidators     37
 1                  Deposit Gatherers        35
 2                  Steady Operators        763
 3                  Digital Retreaters       24

Medium:
innovation_cluster  cluster_name         
-1                  Noise                      77
 0                  Branch Expanders          250
 1                  Deposit Restructurers    3810
 2                  Deposit Transformers      125
 3                  Steady Operators           46
 4                  Relationship Builders      54

Small:
innovation_cluster  cluster_name       
-1                  Noise                   2707
 0                  Rapid Capitalizers       427
 1                  Restructurers            366
 2                  Steady Operators       33577
 3                  Stressed Performers      325


In [15]:
from sklearn.metrics import pairwise_distances
import numpy as np

for tier in ['Large', 'Medium', 'Small']:
    tier_data = df_changes_optimized[df_changes_optimized['bank_tier'] == tier].copy()
    
    noise_mask = tier_data['innovation_cluster'] == -1
    noise_points = tier_data[noise_mask]
    clustered_points = tier_data[~noise_mask]
    
    if len(noise_points) == 0:
        continue
    
    # Compute centroids in UMAP space for each cluster
    centroids = clustered_points.groupby('innovation_cluster')[['umap_1', 'umap_2']].mean()
    
    # Distance from each noise point to each centroid
    noise_coords = noise_points[['umap_1', 'umap_2']].values
    centroid_coords = centroids.values
    centroid_labels = centroids.index.values
    
    dists = pairwise_distances(noise_coords, centroid_coords)
    
    # Find nearest and second nearest
    nearest_idx = dists.argmin(axis=1)
    nearest_cluster = centroid_labels[nearest_idx]
    nearest_dist = dists[np.arange(len(dists)), nearest_idx]
    
    # Second nearest
    dists_masked = dists.copy()
    dists_masked[np.arange(len(dists)), nearest_idx] = np.inf
    second_idx = dists_masked.argmin(axis=1)
    second_cluster = centroid_labels[second_idx]
    second_dist = dists_masked[np.arange(len(dists)), second_idx]
    
    # Map cluster IDs to names
    name_map = dict(zip(
        tier_data.groupby('innovation_cluster')['cluster_name'].first().index,
        tier_data.groupby('innovation_cluster')['cluster_name'].first().values
    ))
    
    print(f"\n{'='*70}")
    print(f"{tier} — Noise point analysis ({len(noise_points)} observations)")
    print(f"{'='*70}")
    
    # Summary: how many noise points are nearest to each cluster
    nearest_names = [name_map.get(c, f'Cluster {c}') for c in nearest_cluster]
    second_names = [name_map.get(c, f'Cluster {c}') for c in second_cluster]
    
    print(f"\nNearest cluster distribution:")
    from collections import Counter
    for name, count in Counter(nearest_names).most_common():
        print(f"  {name:<25} {count:>5} ({count/len(noise_points)*100:.1f}%)")
    
    print(f"\nMost common 'between' pairs:")
    pairs = [tuple(sorted([n1, n2])) for n1, n2 in zip(nearest_names, second_names)]
    for pair, count in Counter(pairs).most_common(5):
        print(f"  {pair[0]} ↔ {pair[1]:<25} {count:>5} ({count/len(noise_points)*100:.1f}%)")
    
    # Show distance ratio — how "torn" each noise point is
    ratio = nearest_dist / second_dist  # closer to 1 = more ambiguous
    print(f"\nAmbiguity (nearest/second dist): "
          f"mean={ratio.mean():.3f}, median={np.median(ratio):.3f}")
    print(f"  Highly ambiguous (ratio > 0.8): {(ratio > 0.8).sum()} ({(ratio > 0.8).mean()*100:.1f}%)")
    print(f"  Clear nearest (ratio < 0.5):    {(ratio < 0.5).sum()} ({(ratio < 0.5).mean()*100:.1f}%)")


Large — Noise point analysis (48 observations)

Nearest cluster distribution:
  Steady Operators             22 (45.8%)
  Branch Consolidators         15 (31.2%)
  Digital Retreaters           11 (22.9%)

Most common 'between' pairs:
  Deposit Gatherers ↔ Steady Operators             17 (35.4%)
  Branch Consolidators ↔ Deposit Gatherers            15 (31.2%)
  Digital Retreaters ↔ Steady Operators             11 (22.9%)
  Branch Consolidators ↔ Steady Operators              5 (10.4%)

Ambiguity (nearest/second dist): mean=0.701, median=0.734
  Highly ambiguous (ratio > 0.8): 1 (2.1%)
  Clear nearest (ratio < 0.5):    5 (10.4%)

Medium — Noise point analysis (77 observations)

Nearest cluster distribution:
  Steady Operators             39 (50.6%)
  Deposit Restructurers        33 (42.9%)
  Relationship Builders         4 (5.2%)
  Deposit Transformers          1 (1.3%)

Most common 'between' pairs:
  Relationship Builders ↔ Steady Operators             40 (51.9%)
  Deposit Restructurer